In [ ]:
import os
import numpy as np
import pandas as pd
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
from sklearn import tree
from sklearn.model_selection import train_test_split
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')

In [ ]:
train_df

In [ ]:
train_df['Destination'].value_counts()

In [ ]:
y = train_df.Transported

заменить пропуски в планете назначения на самую популярную, просуммировать все траты пассажира в одну колонку, удалить ненужные колонки

In [ ]:
train_df = train_df.drop(['Cabin', 'Name', 'Transported'], axis = 1)

In [ ]:
train_df['Destination'] = train_df['Destination'].fillna('TRAPPIST-1e')

In [ ]:
train_df['HomePlanet'] = train_df['HomePlanet'].fillna('Earth')

In [ ]:
train_df['TotalSpent'] = train_df[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis = 1)

In [ ]:
train_df = train_df.drop(['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], axis = 1)

In [ ]:
X = train_df

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state = 42)

In [ ]:
def objective(trial):
    
    n_estimators = trial.suggest_int('n_estimators', 50, 1000, step=50)
    max_depth = trial.suggest_int('max_depth', 2, 32, log=True)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 15)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    
    clf = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf = min_samples_leaf,
        max_features = max_features,
        criterion = criterion,
        random_state = 42,
        n_jobs = -1)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scores = cross_val_score(
        clf, X_train, y_train, 
        cv=cv, 
        scoring='roc_auc',
        n_jobs=-1)
    
    return scores.mean()

study = optuna.create_study(
    study_name='my_rf_study', 
    storage='sqlite:///optuna_study.db',
    load_if_exists=True)

study.optimize(objective, n_trials=100, show_progress_bar=True)

In [ ]:
print("Лучшие гиперпараметры:", study.best_params)

In [ ]:
print("Лучший ROC-AUC:", study.best_value)

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_param_importances(study)